# DiffSinger Miadu Fine-tuning (Colab 斷點續訓版)
**特點**：
- 每 500 步自動保存權重至 Google Drive
- 支持斷點續訓，斷線後重新執行即可從上次位置繼續
- 已對齊 128-bin MIDI 模型架構

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取代碼與環境配置

In [ ]:
%cd /content/
!git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
%cd diffsinger-repo

# 安裝必要依賴 (補齊 pycwt, pyworld 等庫)
!pip install parselmouth pyloudnorm webrtcvad pypinyin lightning-flash==0.7.0 pycwt pyworld scipy==1.9.1

## 第三步：建立安全連接 (軟連結至 Drive)

In [ ]:
# 創建雲端儲存目錄
!mkdir -p "/content/drive/MyDrive/DiffSinger_Colab/checkpoints_finetune"

# 建立軟連結：將本地的 checkpoints 指向雲端硬碟，確保保存時直接入庫
!rm -rf checkpoints
!ln -s "/content/drive/MyDrive/DiffSinger_Colab/checkpoints_finetune" checkpoints

# 初始化目錄
!mkdir -p data/binary

# 搬運預訓練權重與二進位數據 (如果雲端已有則不會重複覆蓋)
import os
if not os.path.exists("checkpoints/1117_opencpop_ds1000_strict_pinyin"):
    print("| 正在從 Drive 獲取基礎權重...")
    !cp -r "/content/drive/MyDrive/DiffSinger_Colab/1117_opencpop_ds1000_strict_pinyin" checkpoints/
    !cp -r "/content/drive/MyDrive/DiffSinger_Colab/nsf_hifigan_44.1k_hop512_128bin_2024.02" checkpoints/

if not os.path.exists("data/binary/miadu_finetune"):
    print("| 正在搬運二進位數據...")
    !cp -r "/content/drive/MyDrive/DiffSinger_Colab/miadu_finetune" data/binary/

print("| 環境就緒！所有訓練進度將實時同步至雲端硬碟。")

## 第四步：啟動訓練 (支持自動斷點續訓)

In [ ]:
import os
os.environ['PYTHONPATH'] = "."

# 注意：這裡去掉了 --reset。
# 如果是「第一次訓練」，它會加載 load_ckpt 指向的 20 萬步模型。
# 如果是「繼續訓練」，它會自動尋找 checkpoints/miadu_finetune_v1 裡的最新權重。
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1